## Clinical Data Preprocessing
Steps include:

1. Cohort definition: filter RA (TACERA), remove (VACCINE) from df_clinical, save as df_clinical_ra_only
2. Raw data audit (df_clinical_ra_only, df_steroids, df_meds) and variance report
3. Individual dataset cleaning (clean each table):
4. a. df_clinical_ra_only: type fixing, standardise binary, missing values, encoding categories (gender, race), keep digest for merging, remove redundancies (inkl. duplicate columns, constant values, highly missing features). Export as df_clinical_ra_only_processed
4. b. df_steroids: aggregate (to a patient level), type fixing, missing values, encoding, etc. Export as df_steroids_processed
4. c. df_meds: aggregate (to a patient level), type fixing, missing values, encoding. Export as df_meds_processed
5. Merge df_clinical_ra_only_processed, df_steroids_processed, df_meds_processed into df_clinical_processed_final (one row per patient, all clinical data in one table)

## Add common imports
From /src/01_common_imports.py and from other relevant /src/ files

In [90]:
from src.common_imports import *
import importlib
import src.reporting as rp
importlib.reload(rp)
from src.reporting import clinical_data_audit, low_variance_report
from src.load_data import load_all_sheets
import src.check_data as check
importlib.reload(check)
from src.check_data import sanity_structure, sanity_contract_check, sanity_numeric, sanity_binary, sanity_categorical, sanity_dates, sanity_duplicates, sanity_check_clinical, check_unique_patients
import src.cleaning as clean
importlib.reload(clean)
from src.cleaning import filter_ra_cohort, clean_patient_level_data, clean_event_level_data, aggregate_steroids, aggregate_meds, merge_patient_datasets
from src.config.data_contract_preprocessing import CLINICAL_CONTRACT, STEROIDS_CONTRACT, MEDS_CONTRACT

## Load All Sheets and extract df_clinical

In [32]:
data = load_all_sheets()
df_clinical = data["df_clinical"]

Loading clinical...
Loading protogen...
Loading somascan...
Done loading


## Remove non-RA patients (those in the VACCINE and not in the TACERA study) and export as new dataframe

In [33]:
df_clinical_ra_only = filter_ra_cohort(df_clinical)
df_clinical_ra_only["Study"].value_counts()


===== RA COHORT FILTER =====
Before filtering: 327 rows
After filtering:  275 rows
Removed (non-RA / other cohorts): 52



Study
TACERA    275
Name: count, dtype: int64

## Clinical Data Audit (RA-only patients) / Report per DataFrame
df_clinical

In [34]:
clinical_data_audit(df_clinical_ra_only, name="dataset")


CLINICAL AUDIT: DATASET

1. STRUCTURE
Shape: 275 rows × 67 columns

2. DUPLICATES
Duplicate rows: 0

3. MISSING VALUES

Columns > 40% missing (likely drop):


,missing_count,missing_pct
Hep B serology wk 9 (IU/mL),275,1.0
vaccine centre,275,1.0



Moderate missing values (likely impute):


,missing_count,missing_pct
FinalxRAYScore,40,0.145455
HEIGHT,7,0.025455
InitialxRAYScore,3,0.010909
Erosive,3,0.010909
Remission month,2,0.007273
Remission(<2.6DAS),2,0.007273
HighDisease(>4DAS),2,0.007273
CRP.9M,1,0.003636



4. NUMERIC FEATURES
Numeric columns: 3

Top features by outliers:


,count,mean,std,min,25%,50%,75%,max,outliers_iqr
Region,275.0,4.403636,3.341946,1.0,1.0,3.0,7.0,10.0,0
Hub,275.0,3.454545,1.866123,1.0,2.0,3.0,5.0,7.0,0
REGION_HUB,275.0,6.083636,3.521092,1.0,3.0,6.0,8.0,12.0,0



5. CATEGORICAL CONSISTENCY
Columns with inconsistent casing: ['Patient_ID', 'Digest', 'Study', 'IM.STEROIDS.3MONTHS', 'ACPA.POSITIVE', 'RHUEMATOID.FACTOR', 'DAS28.0M', 'DAS28.3M', 'DAS28.6M', 'DAS28.9M', 'DAS28.12M', 'DAS28.18M', 'Remission month', 'Remission(<2.6DAS)', 'HighDisease(>4DAS)', 'Symp_Duration', 'HAQ.0M', 'HAQ.6M', 'SDAI.0M', 'SDAI.6M', 'SDAI.12M', 'BASOPHILS.0M', 'EOSINOPHILS.0M', 'HB.0M', 'LYMPHOCYTES.0M', 'MONOCYTES.0M', 'NEUTROPHILS.0M', 'PLT.0M', 'WBC.0M', 'CRP.0M', 'ESR.0M', 'FATIQUE.0M', 'PAIN.0M', 'TOTAL.SWOLLEN.0M', 'TOTAL.TENDER.0M', 'BASOPHILS.6M', 'EOSINOPHILS.6M', 'HB.6M', 'LYMPHOCYTES.6M', 'MONOCYTES.6M', 'NEUTROPHILS.6M', 'PLT.6M', 'WBC.6M', 'CRP.6M', 'FATIQUE.6M', 'PAIN.6M', 'TOTAL.SWOLLEN.6M', 'TOTAL.TENDER.6M', 'CRP.9M', 'TOTAL.SWOLLEN.9M', 'TOTAL.TENDER.9M', 'ORAL.STEROIDS.3M', 'AGE', 'RACE', 'GENDER', 'HEIGHT', 'WEIGHT', 'ALCOHOL_Y_N', 'CURENT SMOKER', 'InitialxRAYScore', 'FinalxRAYScore', 'Erosive']

6. CLEANING SUMMARY
- High missing (> 40%): 2
- Mod

{'high_missing':                              missing_count  missing_pct
 Hep B serology wk 9 (IU/mL)            275          1.0
 vaccine centre                         275          1.0,
 'mid_missing':                     missing_count  missing_pct
 FinalxRAYScore                 40     0.145455
 HEIGHT                          7     0.025455
 InitialxRAYScore                3     0.010909
 Erosive                         3     0.010909
 Remission month                 2     0.007273
 Remission(<2.6DAS)              2     0.007273
 HighDisease(>4DAS)              2     0.007273
 CRP.9M                          1     0.003636}

df_steroids

In [35]:
df_steroids = data["df_steroids"]
clinical_data_audit(df_steroids, name="dataset")


CLINICAL AUDIT: DATASET

1. STRUCTURE
Shape: 414 rows × 10 columns

2. DUPLICATES
Duplicate rows: 0

3. MISSING VALUES

Columns > 40% missing (likely drop):


,missing_count,missing_pct



Moderate missing values (likely impute):


,missing_count,missing_pct
Date of Assessment,2,0.004831



4. NUMERIC FEATURES
Numeric columns: 4

Top features by outliers:


,count,mean,std,min,25%,50%,75%,max,outliers_iqr
Joint Injected,414.0,17.495169,3.959080,3.0,19.0,19.0,19.0,29.0,93
Dose,414.0,100.454831,73.450490,-99.0,80.0,120.0,120.0,1000.0,9
Unit,414.0,1.009662,0.120143,1.0,1.0,1.0,1.0,3.0,3
Assessment,414.0,6.371981,4.282041,3.0,3.0,6.0,9.0,19.0,1



5. CATEGORICAL CONSISTENCY
Columns with inconsistent casing: ['Digest', '4. Has the patient received a steroid injection?', 'Steroid', 'Route']

6. CLEANING SUMMARY
- High missing (> 40%): 0
- Moderate missing: 1
- Numeric features: 4




{'high_missing': Empty DataFrame
 Columns: [missing_count, missing_pct]
 Index: [],
 'mid_missing':                     missing_count  missing_pct
 Date of Assessment              2     0.004831}

df_meds

In [36]:
df_meds = data["df_meds"]
clinical_data_audit(df_meds, name="dataset")


CLINICAL AUDIT: DATASET

1. STRUCTURE
Shape: 2534 rows × 9 columns

2. DUPLICATES
Duplicate rows: 0

3. MISSING VALUES

Columns > 40% missing (likely drop):


,missing_count,missing_pct



Moderate missing values (likely impute):


,missing_count,missing_pct
Dose,6,0.002368
Route,6,0.002368
Frequency,6,0.002368
Date of Assessment,5,0.001973



4. NUMERIC FEATURES
Numeric columns: 3

Top features by outliers:


,count,mean,std,min,25%,50%,75%,max,outliers_iqr
Dose,2528.0,126.882516,258.375293,1.0,5.0,20.0,200.0,2500.0,156
Unit,2534.0,1.004736,0.097224,1.0,1.0,1.0,1.0,3.0,6
Assessment,2534.0,8.838595,4.910288,3.0,6.0,9.0,12.0,19.0,0



5. CATEGORICAL CONSISTENCY
Columns with inconsistent casing: ['Digest', 'RA Medication', 'Frequency', 'Route']

6. CLEANING SUMMARY
- High missing (> 40%): 0
- Moderate missing: 4
- Numeric features: 3




{'high_missing': Empty DataFrame
 Columns: [missing_count, missing_pct]
 Index: [],
 'mid_missing':                     missing_count  missing_pct
 Dose                            6     0.002368
 Route                           6     0.002368
 Frequency                       6     0.002368
 Date of Assessment              5     0.001973}

## Clinical Low Variance Audit / Report per DataFrame
df_clinical

In [37]:
low_variance_report(df_clinical_ra_only, name="dataset")


===== LOW VARIANCE REPORT: DATASET =====

Total numeric features: 3
Low variance threshold: 0.01
Low variance features found: 0

No low-variance features detected.


{'low_variance_features': Series([], dtype: float64),
 'variance': Hub            3.482415
 Region        11.168600
 REGION_HUB    12.398089
 dtype: float64,
 'drop_columns': []}

df_steroids

In [29]:
low_variance_report(df_steroids, name="dataset")


===== LOW VARIANCE REPORT: DATASET =====

Total numeric features: 4
Low variance threshold: 0.01
Low variance features found: 0

No low-variance features detected.


{'low_variance_features': Series([], dtype: float64),
 'variance': Unit                 0.014434
 Joint Injected      15.674311
 Assessment          18.335872
 Dose              5394.974444
 dtype: float64,
 'drop_columns': []}

df_meds

In [30]:
low_variance_report(df_meds, name="dataset")


===== LOW VARIANCE REPORT: DATASET =====

Total numeric features: 3
Low variance threshold: 0.01
Low variance features found: 1

Top low-variance features:
Unit    0.009452
dtype: float64


{'low_variance_features': Unit    0.009452
 dtype: float64,
 'variance': Unit              0.009452
 Assessment       24.110930
 Dose          66757.792286
 dtype: float64,
 'drop_columns': ['Unit']}

## Clean clinical data set (patient level)

In [81]:
df_clinical_clean = clean_patient_level_data(df_clinical_ra_only, CLINICAL_CONTRACT)

sanity_check_clinical(df_clinical_clean, CLINICAL_CONTRACT)


===== SHAPE =====
(275, 57)

===== MISSINGNESS (TOP 10) =====
DAS28.12M         0.389091
BASOPHILS.6M      0.200000
HAQ.6M            0.196364
HB.6M             0.192727
NEUTROPHILS.6M    0.192727
PLT.6M            0.192727
MONOCYTES.6M      0.192727
LYMPHOCYTES.6M    0.192727
EOSINOPHILS.6M    0.192727
WBC.6M            0.192727
dtype: float64

===== DTYPE SUMMARY =====
float64    36
object     14
str         7
Name: count, dtype: int64

===== NUMERIC CHECK =====
DAS28.0M: dtype=float64, NaNs=4
DAS28.3M: dtype=float64, NaNs=26
DAS28.6M: dtype=float64, NaNs=35
DAS28.12M: dtype=float64, NaNs=107
HAQ.0M: dtype=float64, NaNs=27
HAQ.6M: dtype=float64, NaNs=54
SDAI.0M: dtype=float64, NaNs=26
SDAI.6M: dtype=float64, NaNs=52
CRP.0M: dtype=float64, NaNs=3
CRP.6M: dtype=float64, NaNs=36
ESR.0M: dtype=float64, NaNs=3
BASOPHILS.0M: dtype=float64, NaNs=4
EOSINOPHILS.0M: dtype=float64, NaNs=3
HB.0M: dtype=float64, NaNs=2
LYMPHOCYTES.0M: dtype=float64, NaNs=2
MONOCYTES.0M: dtype=float64, NaNs=2
NEU

## Clean steroid data set (event level)

In [74]:
df_steroids_clean = clean_event_level_data(df_steroids, STEROIDS_CONTRACT)

sanity_structure(df_steroids_clean)
sanity_contract_check(df_steroids_clean, STEROIDS_CONTRACT)
sanity_numeric(df_steroids_clean, STEROIDS_CONTRACT)
sanity_binary(df_steroids_clean, STEROIDS_CONTRACT)
sanity_categorical(df_steroids_clean, STEROIDS_CONTRACT)
sanity_dates(df_steroids_clean, STEROIDS_CONTRACT)
sanity_duplicates(df_steroids_clean, ["Digest", "Date Given", "Steroid"])

Shape: (414, 10)

Dtypes:
 Digest                                                         str
Date of Assessment                                  datetime64[ns]
4. Has the patient received a steroid injection?             int64
Assessment                                                   int64
Steroid                                                        str
Dose                                                       float64
Unit                                                           str
Date Given                                          datetime64[ns]
Joint Injected                                                 str
Route                                                          str
dtype: object

Missingness (top 10):
 Date of Assessment                                  0.004831
Digest                                              0.000000
4. Has the patient received a steroid injection?    0.000000
Assessment                                          0.000000
Steroid              

## Clean meds data set (event level)

In [77]:
df_meds_clean = clean_event_level_data(df_meds, MEDS_CONTRACT)

sanity_structure(df_meds_clean)
sanity_contract_check(df_meds_clean, MEDS_CONTRACT)
sanity_numeric(df_meds_clean, MEDS_CONTRACT)
sanity_categorical(df_meds_clean, MEDS_CONTRACT)
sanity_binary(df_meds_clean, MEDS_CONTRACT)
sanity_dates(df_meds_clean, MEDS_CONTRACT)
sanity_duplicates(df_meds_clean, subset=["Digest", "Date Started", "RA Medication"])

df_meds_clean["Assessment"].value_counts().head(10)
df_meds_clean["Unit"].value_counts().head(10)

Shape: (2534, 9)

Dtypes:
 Digest                           str
Date of Assessment    datetime64[us]
Assessment                     int64
RA Medication                    str
Dose                         float64
Unit                           int64
Frequency                        str
Route                            str
Date Started          datetime64[us]
dtype: object

Missingness (top 10):
 Date of Assessment    0.034333
Date Started          0.028019
Route                 0.002368
Frequency             0.002368
Dose                  0.002368
Digest                0.000000
Assessment            0.000000
RA Medication         0.000000
Unit                  0.000000
dtype: float64
Missing expected columns: []
Extra columns: ['Assessment', 'Unit']

Dose
dtype: float64
non-numeric (NaN count): 6
min/max: 1.0 2500.0

RA Medication (top values):
RA Medication
Folic Acid                                   819
Methotrexate                                 818
Hydroxychloroquine              

Unit
1    2528
3       6
Name: count, dtype: int64

## Aggregate event-level data (steroids, meds) to patient level and merge with clinical data to create final clinical dataset for analysis

In [89]:
df_steroids_agg = aggregate_steroids(df_steroids_clean)
df_meds_agg = aggregate_meds(df_meds_clean)

sanity_structure(df_steroids_agg)
sanity_structure(df_meds_agg)

check_unique_patients(df_steroids_agg)
check_unique_patients(df_meds_agg)

Shape: (188, 9)

Dtypes:
 Digest                                str
steroid_injection_count             int64
total_dose                        float64
mean_dose                         float64
max_dose                          float64
intraarticular_count                int64
intramuscular_count                 int64
first_injection            datetime64[ns]
last_injection             datetime64[ns]
dtype: object

Missingness (top 10):
 Digest                     0.0
steroid_injection_count    0.0
total_dose                 0.0
mean_dose                  0.0
max_dose                   0.0
intraarticular_count       0.0
intramuscular_count        0.0
first_injection            0.0
last_injection             0.0
dtype: float64
Shape: (257, 7)

Dtypes:
 Digest                           str
med_event_count                int64
unique_medications             int64
total_dose                   float64
mean_dose                    float64
first_med_date        datetime64[us]
last_med_date   

## Merge all clinical data into one final dataframe (one row per patient, all clinical data in one table)

In [92]:
df_final = merge_patient_datasets(df_clinical_clean, df_steroids_agg, df_meds_agg)

print(df_final.shape)
print(df_final["Digest"].nunique())
print(len(df_final))
df_final["Digest"].duplicated().sum() #Must be 0 for a successful merge to patient level

(275, 71)
275
275


np.int64(0)